In [1]:
!pwd

/home/s.yudin


In [ ]:
"""
Скрипт для загрузки данных с MOEX ISS в Parquet.
"""

import requests
import pandas as pd
import sys
from datetime import datetime, timedelta


In [3]:
# Константы
MOEX_BASE_URL = "https://iss.moex.com/iss"
SECURITIES_ENDPOINT = "/engines/stock/markets/shares/boards/TQBR/securities.json"
TRADES_ENDPOINT_TEMPLATE = "/engines/stock/markets/shares/securities/{secid}/trades.json"

In [4]:
def get_securities_list():
    """Получить список всех акций (SECID) с биржи MOEX."""
    url = MOEX_BASE_URL + SECURITIES_ENDPOINT
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()
    
    # columns - список строк с именами колонок
    columns = data['securities']['columns']
    # data - список массивов значений
    rows = data['securities']['data']
    
    df = pd.DataFrame(rows, columns=columns)
    return df['SECID'].tolist()


In [5]:

# def select_ticker_by_student_name(family_name, secids):
#     """
#     Выбрать тикер на основе фамилии студента.
#     Идея: отсортировать SECID, взять хеш от фамилии по модулю длины списка.
#     Это эмулирует "порядок следования относительно фамилии".
#     """
#     sorted_secids = sorted(secids)
#     hash_val = int(hashlib.md5(family_name.encode()).hexdigest(), 16)
#     index = hash_val % len(sorted_secids)
#     return sorted_secids[index]


In [6]:
def get_trades_for_day(secid, trade_date):
    """
    Получить сделки по тикеру за определённую дату.
    Дата в формате YYYY-MM-DD.
    """
    url = MOEX_BASE_URL + TRADES_ENDPOINT_TEMPLATE.format(secid=secid)
    params = {
        'from': trade_date,
        'till': trade_date
    }
    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()
    
    # Проверяем, есть ли данные
    if 'trades' not in data or 'data' not in data['trades']:
        print(f"Нет данных по сделкам для {secid} на дату {trade_date}")
        return pd.DataFrame()
    
    columns = data['trades']['columns']
    rows = data['trades']['data']
    df = pd.DataFrame(rows, columns=columns)
    return df

In [9]:

# 1. Получить список SECID
print("Загрузка списка акций...")
secids = get_securities_list()
print(f"Получено {len(secids)} акций.")
for _ in secids:
    print(_)


Загрузка списка акций...
Получено 262 акций.
ABIO
ABRD
AFKS
AFLT
AKRN
ALRS
AMEZ
APRI
APTK
AQUA
ARSA
ASSB
ASTR
AVAN
BANE
BANEP
BAZA
BELU
BISVP
BLNG
BRZL
BSPB
BSPBP
BTBR
CARM
CBOM
CHGZ
CHKZ
CHMF
CHMK
CNRU
CNTL
CNTLP
DATA
DELI
DIAS
DIOD
DOMRF
DVEC
DZRD
DZRDP
EELT
ELFV
ELMT
ENPG
ETLN
EUTR
EVRZ
FEES
FESH
FIXR
FLOT
GAZA
GAZAP
GAZC
GAZP
GAZS
GAZT
GCHE
GECO
GEMA
GEMC
GLRX
GMKN
GTRK
HEAD
HIMCP
HNFG
HYDR
IGST
IGSTP
IRAO
IRKT
IVAT
JNOS
JNOSP
KAZT
KAZTP
KBSB
KCHE
KCHEP
KFBA
KGKC
KGKCP
KLSB
KLVZ
KMAZ
KMEZ
KOGK
KRKN
KRKNP
KRKOP
KROT
KROTP
KRSB
KRSBP
KUZB
KZOS
KZOSP
LEAS
LENT
LIFE
LKOH
LMBZ
LNZL
LNZLP
LPSB
LSNG
LSNGP
LSRG
LVHK
MAGE
MAGEP
MAGN
MBNK
MDMG
MFGS
MFGSP
MGKL
MGNT
MGNZ
MGTS
MGTSP
MISB
MISBP
MOEX
MRKC
MRKK
MRKP
MRKS
MRKU
MRKV
MRKY
MRKZ
MRSB
MSNG
MSRS
MSTT
MTLR
MTLRP
MTSS
MVID
NAUK
NFAZ
NKHP
NKNC
NKNCP
NKSH
NLMK
NMTP
NNSB
NNSBP
NSVZ
NVTK
OGKB
OKEY
OMZZP
OZON
OZPH
PAZA
PHOR
PIKK
PLZL
PMSB
PMSBP
POSI
PRFN
PRMB
PRMD
RAGR
RASP
RBCM
RDRB
RENI
RGSS
RKKE
RNFT
ROLO
ROSN
ROST
RTGZ
RTKM
RTKMP
RTSB
RTSBP

In [10]:

# 2. Выбрать тикер по фамилии
# ticker = select_ticker_by_student_name("yudin", secids)
ticker = "YDEX"
print(f"Выбран тикер: {ticker}")


Выбран тикер: YDEX


In [11]:

# 3. Определить дату

# Последний торговый день: сегодня, если сегодня будний день, иначе пятница
today = datetime.now()
if today.weekday() < 5:  # пн-пт
    trade_date = today.strftime('%Y-%m-%d')
else:
    # сдвиг назад до пятницы
    offset = (today.weekday() - 4) % 7
    last_trading_day = today - timedelta(days=offset)
    trade_date = last_trading_day.strftime('%Y-%m-%d')
print(f"Загружаем сделки за дату: {trade_date}")


Загружаем сделки за дату: 2026-06-17


In [12]:
# 4. Получить сделки
df_trades = get_trades_for_day(ticker, trade_date)
if df_trades.empty:
    print("Нет данных, завершение.")
    sys.exit(1)

print(f"Получено {len(df_trades)} записей о сделках.")


Получено 5000 записей о сделках.


In [13]:
local_filename = f"{ticker}_{trade_date}.parquet"
print(local_filename)

YDEX_2026-06-17.parquet


In [14]:
df_trades.to_parquet(local_filename,index=False)

In [ ]:

print("Готово!")

Готово!
